# 0.2 Xatu Calldata Pull

This notebook is the canonical calldata source for the bandwidth pipeline. It pulls total raw calldata bytes from Xatu's canonical beacon execution-payload transaction table, then validates zero/nonzero byte counts from Xatu's execution transaction tables.

Output:

```text
calldata_bytes = sum(canonical_beacon_block_execution_transaction.call_data_size)
calldata_gas = sum(4 * n_input_zero_bytes + 16 * n_input_nonzero_bytes)
```

## Why Xatu for Calldata

Xatu has full payload transaction coverage for calldata, and it is much cheaper to query at scale than RPC. Raw bytes come from the beacon payload transaction table. Zero/nonzero byte counts come from `execution_transaction` only after validating that it matches the beacon payload transaction count and raw byte total. RPC remains the source for BAL bytes because exact BAL reads need `prestateTracer`.

In [1]:
import os
from pathlib import Path

import clickhouse_connect
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim.xatu_calldata import query_xatu_calldata_by_block

load_dotenv(PROJECT_ROOT / ".env")
missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing .env values: " + ", ".join(missing))

RAW_HOST = os.environ.get("CLICKHOUSE_RAW_HOST", "clickhouse-raw.xatu.ethpandaops.io")
CBT_HOST = os.environ.get("CLICKHOUSE_CBT_HOST", "clickhouse-cbt.xatu.ethpandaops.io")

raw_client = clickhouse_connect.get_client(
    host=RAW_HOST,
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)

cbt_client = clickhouse_connect.get_client(
    host=CBT_HOST,
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)
print("raw", RAW_HOST, raw_client.query("SELECT version()").result_rows)
print("cbt", CBT_HOST, cbt_client.query("SELECT version()").result_rows)

raw clickhouse-raw.xatu.ethpandaops.io [('26.2.5.45',)]
cbt clickhouse-cbt.xatu.ethpandaops.io [('26.2.5.45',)]


In [2]:
NETWORK = "mainnet"
START_BLOCK = 24_120_001
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
WRITE_CSV = True

In [3]:
calldata = query_xatu_calldata_by_block(raw_client, BLOCKS, network=NETWORK)

cbt_block_size_gas = cbt_client.query_df(
    f"""
    SELECT
        block_number,
        gas_block_size AS cbt_gas_block_size
    FROM {NETWORK}.int_block_resource_gas
    WHERE block_number IN {{blocks:Array(UInt64)}}
    ORDER BY block_number
    """,
    parameters={"blocks": BLOCKS},
)

calldata = calldata.merge(cbt_block_size_gas, on="block_number", how="left")
calldata["cbt_gas_block_size"] = calldata["cbt_gas_block_size"].astype("Int64")
calldata["calldata_gas_minus_cbt_gas_block_size"] = (
    calldata["calldata_gas"].astype("Int64") - calldata["cbt_gas_block_size"]
)
calldata["calldata_gas_matches_cbt_gas_block_size"] = (
    calldata["calldata_gas"].notna()
    & calldata["cbt_gas_block_size"].notna()
    & (calldata["calldata_gas"].astype("Int64") == calldata["cbt_gas_block_size"])
)

calldata["execution_tx_row_delta"] = calldata["execution_tx_rows"] - calldata["n_txs_from_payload"]
calldata["execution_calldata_delta"] = calldata["execution_calldata_bytes"] - calldata["calldata_bytes"]

display(calldata)

summary = pd.DataFrame([{
    "blocks_checked": len(calldata),
    "execution_matches": int(calldata["execution_matches_beacon"].sum()),
    "total_payload_txs": int(calldata["n_txs_from_payload"].sum()),
    "total_calldata_bytes": int(calldata["calldata_bytes"].sum()),
    "total_zero_bytes": int(calldata["calldata_zero_bytes"].dropna().sum()),
    "total_nonzero_bytes": int(calldata["calldata_nonzero_bytes"].dropna().sum()),
    "total_calldata_gas": int(calldata["calldata_gas"].dropna().sum()),
    "total_cbt_gas_block_size": int(calldata["cbt_gas_block_size"].dropna().sum()),
    "gas_block_size_matches": int(calldata["calldata_gas_matches_cbt_gas_block_size"].sum()),
    "gas_block_size_mismatches": int((~calldata["calldata_gas_matches_cbt_gas_block_size"]).sum()),
    "total_calldata_gas_minus_cbt": int(calldata["calldata_gas_minus_cbt_gas_block_size"].dropna().sum()),
    "mean_calldata_bytes_per_block": calldata["calldata_bytes"].mean(),
    "mean_calldata_gas_per_block": calldata["calldata_gas"].dropna().mean(),
}])
display(summary)

mismatches = calldata[~calldata["calldata_gas_matches_cbt_gas_block_size"]][
    [
        "block_number",
        "calldata_gas",
        "cbt_gas_block_size",
        "calldata_gas_minus_cbt_gas_block_size",
        "calldata_gas_source",
        "execution_matches_beacon",
    ]
]
display(mismatches)

if WRITE_CSV:
    data_dir = PROJECT_ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    out = data_dir / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
    calldata.to_csv(out, index=False)
    print(out)

,block_number,slot,n_txs_from_payload,n_txs,calldata_bytes,execution_tx_rows,execution_calldata_bytes,calldata_zero_bytes,calldata_nonzero_bytes,calldata_gas,...,execution_rows_n_input_positive,execution_matches_beacon,blob_versioned_hash_count,blob_versioned_hash_bytes,blob_versioned_hash_gas,cbt_gas_block_size,calldata_gas_minus_cbt_gas_block_size,calldata_gas_matches_cbt_gas_block_size,execution_tx_row_delta,execution_calldata_delta
0,24120001,13350651,344,344,238525,344,238525,144495,94030,2082460,...,237,True,13,416,26624,1960800,121660,False,0,0
1,24120002,13350652,179,179,44492,179,44492,29852,14640,353648,...,117,True,6,192,12288,1020300,-666652,False,0,0
2,24120003,13350653,489,489,136567,489,136567,86377,50190,1148548,...,377,True,4,128,8192,2787300,-1638752,False,0,0
3,24120004,13350654,232,232,70460,232,70460,44058,26402,598664,...,166,True,9,288,18432,1322400,-723736,False,0,0
4,24120005,13350655,250,250,46996,250,46996,30210,16786,389416,...,163,True,0,0,0,1425000,-1035584,False,0,0
5,24120006,13350656,242,242,101974,242,101974,57426,44548,942472,...,164,True,5,160,10240,1379400,-436928,False,0,0
6,24120007,13350657,69,69,12220,69,12220,6755,5465,114460,...,38,True,1,32,2048,393300,-278840,False,0,0
7,24120008,13350658,483,483,133308,483,133308,89789,43519,1055460,...,298,True,8,256,16384,2753100,-1697640,False,0,0
8,24120009,13350659,256,256,91881,256,91881,62112,29769,724752,...,165,True,0,0,0,1459200,-734448,False,0,0
9,24120010,13350660,416,416,145880,416,145880,81120,64760,1360640,...,298,True,5,160,10240,2371200,-1010560,False,0,0


,blocks_checked,execution_matches,total_payload_txs,total_calldata_bytes,total_zero_bytes,total_nonzero_bytes,total_calldata_gas,total_cbt_gas_block_size,gas_block_size_matches,gas_block_size_mismatches,total_calldata_gas_minus_cbt,mean_calldata_bytes_per_block,mean_calldata_gas_per_block
0,50,50,14989,5412521,3378185,2034336,46062116,85437300,0,50,-39375184,108250.42,921242.32


,block_number,calldata_gas,cbt_gas_block_size,calldata_gas_minus_cbt_gas_block_size,calldata_gas_source,execution_matches_beacon
0,24120001,2082460,1960800,121660,execution_transaction,True
1,24120002,353648,1020300,-666652,execution_transaction,True
2,24120003,1148548,2787300,-1638752,execution_transaction,True
3,24120004,598664,1322400,-723736,execution_transaction,True
4,24120005,389416,1425000,-1035584,execution_transaction,True
5,24120006,942472,1379400,-436928,execution_transaction,True
6,24120007,114460,393300,-278840,execution_transaction,True
7,24120008,1055460,2753100,-1697640,execution_transaction,True
8,24120009,724752,1459200,-734448,execution_transaction,True
9,24120010,1360640,2371200,-1010560,execution_transaction,True


/Users/william/PycharmProjects/eip-7999-research/data/xatu_calldata_24120001_24120050.csv
